In [ ]:
from langgraph.graph import StateGraph, END, START
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [ ]:
load_dotenv()

In [ ]:
model = ChatOpenAI(model = 'GPT-4o-mini')

In [ ]:
# here for the sentiment analysis we will get the structured output from the LLM , and for that we have to to use 
# Schema , therefore we make the class by using the Schema  only
class Sentimentschema(BaseModel):
    sentiment : Literal['positive','negative'] = Field(description='Sentiment of the review')

In [ ]:
structured_model = model.with_structured_output(Sentimentschema)

In [ ]:
prompt = 'What is the sentiment of the review - Himkesh is the King '
structured_model.invoke(prompt).sentiment

In [ ]:
class reviewstate(TypedDict):
    
    review: str
    sentiment : Literal['positive','negative']
    diagnosis : str
    response : str

In [ ]:
def find_sentiment (state: reviewstate):
    
    prompt = f'For the following statement find the sentiment , the statement is - {state['review']}'
    sentiment = structured_model.invoke(prompt).sentiment
    
    return {'sentiment':sentiment}

def check_condition(state : reviewstate) -> Literal['run_diagnosis','positive_response']
    if state['sentiment'] =='positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

def run_diagnosis(state : reviewstate):
    
    prompt = f'Find out the ways how can we solve the problem of the user , what all potential solution will be for the problem stated by the user in his following reivew - {state['review']}'
    state['diagnosis'] = model.invoke(prompt).result
    
    diagnosis = state['diagnosis']
    
    return {'diagnosis' : diagnosis}

def negative_response(state : reviewstate)

In [ ]:
graph = StateGraph(reviewstate)

#create nodes
graph.add_node('find_sentiment', find_sentiment)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('positive_response', positive_response)
graph.add_node('negative_response', negative_response)

#create the edges
graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_condition)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('negative_response',END)
graph.add_edge('positive_response',END)

workflow = graph.compile()

In [ ]:
workflow

In [ ]:
initial_state = {
    
}